# Reel Factory — Gemini + HyperFrames + Remotion

Upload long-form podcast/interview videos to Google Drive. This notebook transcribes them, ranks candidate moments, waits for your selection, then renders vertical reels and saves them back to Drive.

**Runtime:** choose a T4 GPU in Runtime → Change runtime type before running setup.

In [ ]:
import os, sys, subprocess, pathlib, json, getpass, shutil

REPO_URL = 'https://github.com/martyroy6996-spec/remotion-shorts-engine.git'
REPO_DIR = pathlib.Path('/content/remotion-shorts-engine')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
os.chdir(REPO_DIR)

print('Installing Node.js 22 and FFmpeg...')
subprocess.run('curl -fsSL https://deb.nodesource.com/setup_22.x | bash -', shell=True, check=True)
subprocess.run(['apt-get', 'install', '-y', 'nodejs', 'ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['npm', 'install'], check=True)
print('Setup complete.')

In [ ]:
import json, pathlib, subprocess, torch

os.chdir(REPO_DIR)
print('Node:', subprocess.check_output(['node', '--version'], text=True).strip())
print('npm:', subprocess.check_output(['npm', '--version'], text=True).strip())

# Ensure dependencies are installed in this repository, not fetched ad hoc by npx.
print('Checking project dependencies...')
subprocess.run(['npm', 'install'], cwd=REPO_DIR, check=True)

package_data = json.loads((REPO_DIR / 'package.json').read_text(encoding='utf-8'))
for package_name in ['remotion', '@remotion/cli', 'hyperframes']:
    package_file = REPO_DIR / 'node_modules' / pathlib.Path(*package_name.split('/')) / 'package.json'
    if not package_file.exists():
        raise RuntimeError(f'Missing dependency: {package_name}. Check the npm install output above.')
    installed = json.loads(package_file.read_text(encoding='utf-8'))
    print(f"{package_name}: {installed.get('version', 'version unknown')}")

for binary_name in ['remotion', 'hyperframes']:
    binary_path = REPO_DIR / 'node_modules' / '.bin' / binary_name
    if not binary_path.exists():
        raise RuntimeError(f'Missing local executable: {binary_path}. npm install did not finish correctly.')
    print(f'{binary_name} executable: OK')

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No CUDA GPU detected. Check Runtime → Change runtime type → T4 GPU.')

print('Dependency verification complete.')

## Connect Google Drive

The notebook creates ReelFactory/input and ReelFactory/output in your Drive. Put your source videos in the input folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/ReelFactory')
INPUT_DIR = DRIVE_ROOT / 'input'
OUTPUT_DIR = DRIVE_ROOT / 'output'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Input folder:', INPUT_DIR)
print('Output folder:', OUTPUT_DIR)

## Gemini key and model settings

Recommended: add GEMINI_API_KEY in Colab Secrets and enable notebook access. The hidden prompt below is a session-only fallback; the key is not saved to GitHub.

In [ ]:
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = None
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass('Paste Gemini API key (hidden; session only): ')
if not GEMINI_API_KEY:
    raise RuntimeError('Gemini API key is required.')
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

GEMINI_MODEL = 'gemini-3.8-flash'
WHISPER_MODEL = 'small'  # Use medium or large-v3-turbo for heavier transcription.
print('Gemini model:', GEMINI_MODEL)
print('Whisper model:', WHISPER_MODEL)

## Choose a source video

Upload the podcast/interview video to ReelFactory/input, then run this cell. If there are multiple videos, enter the number of the one to process.

In [ ]:
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.webm', '.m4v', '.avi'}
source_files = sorted([p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS])
if not source_files:
    raise FileNotFoundError('No video found. Upload an MP4/MOV/MKV/WEBM file to ' + str(INPUT_DIR))
for i, path in enumerate(source_files):
    print(f'[{i}] {path.name}  ({path.stat().st_size / (1024**3):.2f} GB)')
selected_file_index = int(input('Choose video number: ')) if len(source_files) > 1 else 0
SOURCE_VIDEO = source_files[selected_file_index]
WORK_DIR = pathlib.Path('/content/reel-factory-work') / SOURCE_VIDEO.stem
WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT = OUTPUT_DIR / SOURCE_VIDEO.stem
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print('Selected:', SOURCE_VIDEO)

## Analyze and rank moments

Whisper creates word-level timestamps; Gemini ranks moments against the transcript. This step does not render or publish anything.

In [ ]:
analysis_command = [sys.executable, 'pipeline/clipper.py', 'analyze', '--input', str(SOURCE_VIDEO), '--workdir', str(WORK_DIR), '--whisper-model', WHISPER_MODEL, '--gemini-model', GEMINI_MODEL]
subprocess.run(analysis_command, cwd=REPO_DIR, check=True)
analysis = json.loads((WORK_DIR / 'candidates.json').read_text(encoding='utf-8'))
print('\nReview these before rendering:\n')
for i, clip in enumerate(analysis['candidates']):
    print(f"[{i}] Score {clip['score']}/100 | {clip['start_sec']:.1f}s–{clip['end_sec']:.1f}s | {clip['title']}\n    Hook: {clip['hook']}\n    Why: {clip['reason']}\n")

## Select and render

Enter candidate indexes separated by commas (for example 0,2). Rendering takes place in the Colab runtime. The finished MP4s and review metadata are copied to Drive.

In [ ]:
selection = input('Candidate indexes to render (comma-separated; blank = top candidate): ').strip()
selected_indexes = [0] if not selection else sorted(set(int(v.strip()) for v in selection.split(',') if v.strip()))
for candidate_index in selected_indexes:
    render_command = [
        sys.executable, 'pipeline/clipper.py', 'render',
        '--input', str(SOURCE_VIDEO),
        '--workdir', str(WORK_DIR),
        '--candidate-index', str(candidate_index),
        '--drive-output', str(DRIVE_OUTPUT)
    ]
    subprocess.run(render_command, cwd=REPO_DIR, check=True)
print('\nFiles saved to:', DRIVE_OUTPUT)
for output_file in sorted(DRIVE_OUTPUT.glob('*.mp4')):
    print(output_file.name, round(output_file.stat().st_size / (1024**2), 1), 'MB')